In [0]:
%pip install openai
%pip install langchain-text-splitters

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.5/580.5 kB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 41.4 MB/s  0:00:00
  Attempting uninstall: jsonpointer
    Found existing installation: jsonpointer 3.1.1
    Not uninstalling jsonpointer at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4f90126a-7484-4a83-9ef8-cfdba1e43f00
    Can't uninstall 'jsonpointer'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [langchain-text-splitters]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from pyspark.sql.functions import *    
from pyspark.sql.types import *
from datetime import * 
from openai import OpenAI
import os
import json
from pyspark.sql import SparkSession

In [0]:
df_customer = spark.read\
    .option("header" , True)\
    .option("inferschema" , True)\
    .csv("/Volumes/retail_data/bronze/raw_data_volume/gen_ai_datasets/customer_data_genai.csv")

df_employee = spark.read\
    .option("header" , True)\
    .option("inferschema" , True)\
    .csv("/Volumes/retail_data/bronze/raw_data_volume/gen_ai_datasets/employee_data_genai_final.csv")

df_payment  =  spark.read\
    .option("header" , True)\
    .option("inferschema" , True)\
    .csv("/Volumes/retail_data/bronze/raw_data_volume/gen_ai_datasets/payment_data_genai.csv")

In [0]:
df_customer_updated = df_customer.withColumn("salary" , expr("try_cast(salary as int)"))\
                         .withColumn("age" , expr("try_cast(age as int)"))\
                         .withColumn("created_at" , expr("try_cast(created_at as timestamp)"))\
                         .withColumn("email" , expr("try_cast(email as string)"))\
                         .withColumn("Time_stamp" , current_timestamp())

df_employee_updated = df_employee.withColumn("salary" , expr("try_cast(salary as int)"))\
                         .withColumn("manager_id" , expr("try_cast(manager_id as int)"))\
                         .withColumn("joining_date" , expr("try_cast(joining_date as date)"))\
                         .withColumn("bonus" , expr("try_cast(bonus as int)"))\
                         .withColumn("Time_stamp" , current_timestamp())

df_payment_updated  = df_payment.withColumn("amount" , expr("try_cast(amount as int)"))\
                       .withColumn("payment_date" , expr("try_cast(payment_date as timestamp)"))\
                       .withColumn("Time_stamp" , current_timestamp())

In [0]:
a1 = df_customer_updated.count()
a2 = df_customer_updated.dropna(subset = ["customer_id" , "email" , "name"]).count()

b1 = df_employee_updated.count()
b2 = df_employee_updated.dropna(subset = ["emp_id" , "emp_name" , "department"]).count()

c1 = df_payment_updated.count()
c2 = df_payment_updated.dropna(subset = ["payment_id" , "transaction_reference" , "payment_status" , "customer_id"]).count()

print(a1 , f" , Bad Records of Customer data: {a1-a2}" , f" , After Bad records Removed {a2}")
print(b1 , f" , Bad Records of Employee data  : {b1-b2}" , f" , After Bad records Removed {b2}")
print(c1 , f" , Bad Records of Employee Data : {c1-c2}" , f" , After Bad records Removed {c2}")


1520  , Bad Records of Customer data: 42  , After Bad records Removed 1478
1500  , Bad Records of Employee data  : 4  , After Bad records Removed 1496
2000  , Bad Records of Employee Data : 41  , After Bad records Removed 1959


In [0]:

############### Quarantine Bad Records ####################################

df_customer_quarantine = df_customer_updated.filter(col("customer_id").isNull() \
                                                 | col("email").isNull() \
                                                 | col("name").isNull())

df_employee_quarantine = df_employee_updated.filter(col("emp_id").isNull() \
                                                  | col("emp_name").isNull() \
                                                  | col("department").isNull())

df_payment_quarantine = df_payment_updated.filter(col("payment_id").isNull() \
                                                | col("transaction_reference").isNull()\
                                                | col("payment_status").isNull()
                                                | col("customer_id").isNull())



####################### Quarantine data write in bronze table #############################

df_customer_quarantine.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("bronze.quarantine.customer_quarantine")

df_employee_quarantine.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("bronze.quarantine.employee_quarantine")

df_payment_quarantine.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("bronze.quarantine.payment_quarantine")



################## Clean data write in silver table #############################33

df_customer_cleaned = df_customer_updated.dropna(subset = ["customer_id" , "email" , "name"])
df_employee_cleaned = df_employee_updated.dropna(subset = ["emp_id" , "emp_name" , "department"])
df_payment_cleaned = df_payment_updated.dropna(subset = ["payment_id" , "transaction_reference" , "payment_status" , "customer_id"])


df_customer_cleaned.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("silver.cleaned.customer_cleaned")

df_employee_cleaned.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("silver.cleaned.employee_cleaned")

df_payment_cleaned.write\
    .mode("overwrite")\
    .option("mergeSchema" , True)\
    .format("delta")\
    .saveAsTable("silver.cleaned.payment_cleaned")


In [0]:
df_customer_cleaned = spark.table(f"silver.cleaned.customer_cleaned")

df_customer_cleaned = df_customer_cleaned.withColumn("Loan_Account_No",
     when(
        regexp_extract(col("customer_message"), r'((?:BLU|PLU|DIG|LAP|SCF)\d+)', 1) != "",
        regexp_extract(col("customer_message"), r'((?:BLU|PLU|DIG|LAP|SCF)\d+)', 1)
    ).otherwise(None)
).drop("loan_account_number_extracted" , "feedback_sentiment")


df_customer_cleaned = df_customer_cleaned.withColumn("feedback_sentiment" , lit(None).cast("string"))

### Gen AI phase start


In [0]:
from openai import OpenAI

api_key = dbutils.secrets.get(
    scope = "genai-secrets" , 
    key = "OPENAI_API_KEY"
)

client  = OpenAI(api_key = api_key)


### Customer data

In [0]:
feedback =  [f.feedback_text for f in df_customer_cleaned.select("feedback_text").select("feedback_text").distinct().collect()]

response = client.responses.create(
    model = "gpt-5" , 
    input =  f"""
    You are a customer feedback sentiment analyser.

    Classify each feedback into exactly one category : Positive , Negative , or Neutral

    Return the result as Json Array . 
    Do no skip any feedback .

    Requiored Json Format :
    [{{
        "feedback_text" : "Original Feedback" ,
        "sentiment" : "Positive"
    }}]

    Feedbacks : 
    {json.dumps(feedback)}
    """
)
# print(response.output_text)

sentiment  =  json.loads(response.output_text)

for s in sentiment:
    df_customer_cleaned = df_customer_cleaned.withColumn(
        "feedback_sentiment" ,  when(col("feedback_text") == s["feedback_text"] , lit(s["sentiment"])).otherwise(col("feedback_sentiment")))
    
# df_customer_cleaned.display()

# df_customer_cleaned.dropna().groupBy("city").pivot("feedback_sentiment").count().display()

df_customer_cleaned.write\
    .mode("overwrite")\
    .option("mergeschema" , True)\
    .format("delta")\
    .saveAsTable("gold.cleaned.customer_cleaned")

### Payment data

In [0]:
pay = [f.failure_reason_description for f in df_payment_cleaned.select("failure_reason_description").distinct().collect()]

valid_reasons  = [reason for reason in pay if reason is not None]


batch_size = 5 
payment_analysis  = []

for i in range(0 , len(valid_reasons) , batch_size) :

    batch = valid_reasons[i : i + batch_size]

    prompt =  f"""
        You are an Expert payment failure Resaon analyst working for a fintech company . 
        Analyse each paymnet failure description and generate the following information .
        1.Failure_catrgory : 
            classify the failure into a suitable category such as :
            Insufficient Balance , Gateway Timeout , Invalid Payment Details ,
            OTP failure , Network Failure , Expired Documents , OTP failure , Network Failure , Risk Verification , Bank ServerIssue , Transaction Limit , Bank Declined , Missing Information , Unknown User . 
        2.Failure_Severity :
            Classify as High , medium , or Low 
        3.Customer_Action_required :
            Return Yes or No 
        
        4.Resolution_Suggestion :
            Provide a practical solution to resolve the payment failure.
        Important instructions :
            -Analyse every failure description independently . 
            - Do not skip any description .
            - Do not assume information that is not provided . 
            - return only valid Json . 
            - If the reason is missing or unknown , classify it appropriately . 
        Required JSON Format :
            [{{
                    "failure_reason_description" : "original description ,
                    "failure_category" : "Insufficient_Balance" , 
                    "failure_severity" : "Medium" , 
                    "customer_action_required" : "Yes" ,
                    "Resolution_Suggestion" : "Provide a practical solution to resolve the payment failure" , 
                    "Failure_Summary" : "Payment Failed due to insufficient Balance."
                }}]
        Payment Failure Description :
        {json.dumps(batch)}
        """  
        
    response = client.responses.create(
            model = "gpt-5" , 
            input = prompt , 
            max_output_tokens = 4000 )
        
    batch_result = json.loads(response.output_text)

    payment_analysis.extend(batch_result)
        

# print(payment_analysis)

spark = SparkSession.builder.getOrCreate()  

df_payment_analysis =  spark.createDataFrame(payment_analysis)

df_payment_gold  =  df_payment_cleaned.join(df_payment_analysis , on = "failure_reason_description" , how = "left")

df_payment_gold.write\
    .mode("overwrite")\
    .option("mergeschema" , True)\
    .format("delta")\
    .saveAsTable("gold.cleaned.payment_gold")


In [0]:
df_employee_cleaned  =  spark.table(f"""silver.cleaned.employee_cleaned""")

In [0]:
for c in ["skills_description", "employee_feedback", "employee_query"]:
    print(c, df_employee_cleaned.select(c).distinct().count())

skills_description 8
employee_feedback 10
employee_query 8


In [0]:
query = [r for r in df_employee_cleaned.select("skills_description" , "employee_feedback" , "employee_query").distinct().collect()]

query = [{
    "skills_description" : q.skills_description , 
    "employee_query" : q.employee_query , 
    "employee_feedback" : q.employee_feedback
}  
         for q in query ]

print(query)

In [0]:
# df_employee_cleaned  =  spark.table(f"""silver.cleaned.employee_cleaned""")

query =  [f.employee_query  for  f  in  df_employee_cleaned.select("employee_query").distinct().collect()]

query = [q for q in query if q is not None or q != ""]

batch_size = 5 
employee_analysis = []

for i in range(0 , len(query) , batch_size) :

    batch = query[i : i + batch_size]

    prompt  =  f"""
    You are an HR query classification assistant . 

    classify each employee query into one category :
    HR , Salary & compensation , Leave & attendance , 
    Technical Support , Work related , payroll , other . 

    Return only a valid JSON array :

    each object must contain :
    employee_query 
    queety_category

    queries : 
    {json.dumps(batch)}
    """

    response = client.responses.create(
        model = "gpt-5" , 
        input= prompt , 
        max_output_tokens = 2000
    )

    batch_result = json.loads(response.output_text)

    employee_analysis.extend(batch_result)


# print(employee_analysis)

spark = SparkSession.builder.getOrCreate()  

df_employee_analysis =  spark.createDataFrame(employee_analysis)

df_employee_gold  =  df_employee_cleaned.join(df_employee_analysis , on = "employee_query" , how = "left")

df_employee_gold.write\
    .mode("overwrite")\
    .option("overwriteSchema" , "true")\
    .saveAsTable("gold.cleaned.employee_gold")